# 4.4 Streaming: Concepts Notebook

**Generative AI Foundations · Module 4**

| # | Concept |
|---|---|
| 1 | Bucket vs tap: why stream |
| 2 | Chunks |
| 3 | Joining chunks with `+` |
| 4 | From chunk to message |
| 5 | Is the reply complete? The finish reason |
| 6 | Failed turns: what to keep, what to drop |
| 7 | Streaming into chat history |

Some sections build chunks **by hand**, so you can see exactly what happens, even without a model. The others use Ollama with `gemma3:1b`.

## 0. Setup

In [ ]:
%pip install -q "langchain[ollama]"

In [ ]:
import time

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessageChunk, message_chunk_to_message

llm = init_chat_model("ollama:gemma3:1b", temperature=0.7)
print("Ready")

---
## 1. Bucket vs Tap

> **Mental Model: The Bucket and the Tap**
> `invoke` fills a whole bucket before handing it over. `stream` is a tap: water flows as soon as it's ready.

```text
invoke:  [ ......... waiting ......... ] "Once upon a time in Lahore..."
stream:  "Once" " upon" " a" " time" " in" " Lahore" ...
```

In [ ]:
question = "Tell me a two-sentence story about a coder in Lahore."

start = time.perf_counter()
reply = llm.invoke(question)
print(f"invoke: everything after {time.perf_counter() - start:.2f} s")

start = time.perf_counter()
first = None
for chunk in llm.stream(question):
    if first is None and chunk.text:
        first = time.perf_counter() - start
print(f"stream: first words after {first:.2f} s, all done after {time.perf_counter() - start:.2f} s")

Total time is about the same. Streaming just **starts showing** text much sooner.

---
## 2. Chunks

Each piece from `stream` is an `AIMessageChunk`. We print a `|` after each one to see the pieces:

In [ ]:
count = 0
for chunk in llm.stream("In one sentence, what is Python?"):
    print(chunk.text, end="|", flush=True)
    count += 1

print(f"\n\n{count} chunks. Each one is a {type(chunk).__name__}.")

| Part | Meaning |
|---|---|
| `end="\|"` | Print a bar instead of a new line (use `end=""` in real apps) |
| `flush=True` | Show the text **now**, don't wait |

---
## 3. Joining Chunks with `+`

> **Mental Model: Building a Train**
> Each chunk is a carriage. `+` couples the next one on. The last carriage carries the "end of train" sign (the finish reason) and the ticket count (tokens).

Let's build a train **by hand**:

In [ ]:
pieces = [
    AIMessageChunk(content="Once "),
    AIMessageChunk(content="upon "),
    AIMessageChunk(content="a time."),
    AIMessageChunk(content="", response_metadata={"done_reason": "stop"},
                   usage_metadata={"input_tokens": 12, "output_tokens": 4, "total_tokens": 16}),
]

full = None
for piece in pieces:
    full = piece if full is None else full + piece
    print(f"full.text = {full.text!r}")

print("\nFinish reason:", full.response_metadata.get("done_reason"))
print("Tokens:       ", full.usage_metadata)

`+` joins the text **and** keeps the metadata from the last chunk. The same pattern with a real model:

In [ ]:
full = None
for chunk in llm.stream("In one sentence, what is an API?"):
    print(chunk.text, end="", flush=True)
    full = chunk if full is None else full + chunk

print("\n\nTokens:", full.usage_metadata)

---
## 4. From Chunk to Message

The joined result is still a **chunk**. For chat history, turn it into a normal `AIMessage`:

In [ ]:
message = message_chunk_to_message(full)

print("Before:", type(full).__name__, "| type =", full.type)
print("After: ", type(message).__name__, "     | type =", message.type)
print("Text is the same:", message.text == full.text)

---
## 5. Is the Reply Complete?

A complete reply always ends with a **finish reason**. If the connection is cut, the stream can end **quietly, with no error**, and the finish reason is missing.

> **Mental Model: The End-of-Train Sign**
> A train without its last carriage looks fine from the platform, but it has no "end" sign. Always check for the sign.

In [ ]:
def finished(message) -> bool:
    metadata = message.response_metadata
    return bool(metadata.get("finish_reason") or metadata.get("done_reason") or metadata.get("stop_reason"))


complete = AIMessageChunk(content="Once upon a time.") + AIMessageChunk(content="", response_metadata={"done_reason": "stop"})
cut_off = AIMessageChunk(content="Once upon") + AIMessageChunk(content=" a")   # the last carriage never came

print("complete:", repr(complete.text), "->", finished(complete))
print("cut off: ", repr(cut_off.text), "    ->", finished(cut_off))

---
## 6. Failed Turns: What to Keep, What to Drop

A stream can end in four ways:

| Ending | What happens | Keep the reply? |
|---|---|---|
| Normal | All chunks arrive, with a finish reason | Yes |
| Error | An exception is raised | No |
| User presses Ctrl+C | `KeyboardInterrupt` | No |
| Silent cut | Loop ends, no finish reason | No |

> **Mental Model: The Torn Page**
> If a page tears while printing, you don't file the half page. You throw it away and print again.

```text
[system, Q1, A1]  + Q2  ->  stream breaks  ->  drop Q2 and the partial reply  ->  [system, Q1, A1]
```

Below, **pretend streams** let us test all four endings without breaking a real server:

In [ ]:
def normal_stream():
    yield AIMessageChunk(content="Python is ")
    yield AIMessageChunk(content="great.")
    yield AIMessageChunk(content="", response_metadata={"done_reason": "stop"})

def error_stream():
    yield AIMessageChunk(content="Python is ")
    raise ConnectionError("network lost")

def ctrl_c_stream():
    yield AIMessageChunk(content="Python is ")
    raise KeyboardInterrupt   # as if the user pressed Ctrl+C

def silent_cut_stream():
    yield AIMessageChunk(content="Python is ")
    yield AIMessageChunk(content="gr")   # ends here, no finish reason


def run_turn(history, question, stream):
    history.append({"role": "user", "content": question})
    full = None
    try:
        for chunk in stream:
            full = chunk if full is None else full + chunk
    except KeyboardInterrupt:
        history.pop()
        return "stopped by user: turn dropped"
    except Exception as error:
        history.pop()
        return f"error ({type(error).__name__}): turn dropped"

    if full is None or not finished(full):
        history.pop()
        return "cut off (no finish reason): turn dropped"

    history.append(message_chunk_to_message(full))
    return "complete: reply saved"


for name, stream in [("normal", normal_stream()), ("error", error_stream()),
                     ("ctrl+c", ctrl_c_stream()), ("silent cut", silent_cut_stream())]:
    history = [{"role": "system", "content": "Be brief."}]
    result = run_turn(history, "What is Python?", stream)
    print(f"{name:<11} -> {result:<42} history length = {len(history)}")

Only the normal ending grows the history (system + question + reply = 3). Every failure leaves it **exactly as before** (1).

---
## 7. Streaming into Chat History

Now the real thing: stream two turns, save each complete reply, and check the memory.

In [ ]:
history = [{"role": "system", "content": "You are Cue. Keep answers short."}]

for question in ["Hi, my name is Ali.", "What is my name?"]:
    history.append({"role": "user", "content": question})
    print(f"You: {question}\nCue: ", end="")

    full = None
    for chunk in llm.stream(history):
        print(chunk.text, end="", flush=True)
        full = chunk if full is None else full + chunk
    print()

    if full is not None and finished(full):
        history.append(message_chunk_to_message(full))
    else:
        history.pop()

print("\nMessages in history:", [type(m).__name__ if not isinstance(m, dict) else m["role"] for m in history])

---
## Summary

| Concept | Remember |
|---|---|
| Why stream | Same total time; first words appear much sooner |
| Chunks | `llm.stream()` yields `AIMessageChunk` pieces; print `chunk.text` with `end=""`, `flush=True` |
| Joining | `full = chunk if full is None else full + chunk` |
| Chunk to message | `message_chunk_to_message(full)` before saving to history |
| Complete? | Only if there is a finish reason |
| Failed turn | Error, Ctrl+C, or silent cut: drop the question and the partial reply |

## Practice

1. In Section 2, change `"|"` to `""` and add a word counter that prints the total at the end.
2. In Section 6, add a fifth pretend stream that yields **no chunks at all**. What does `run_turn` return?
3. In Section 7, add a third question, then print how many tokens the last reply used.